In [69]:
from pathlib import Path
import gzip
import scipy.io
import anndata as ad
import pandas as pd
import scanpy as sp
import numpy as np
import scipy as sc
from sklearn.model_selection import train_test_split


# 1.) Load Data

In [26]:
metadata = pd.read_csv(
    "GSE164378_sc.meta.data_3P.csv.gz",
    index_col=0,
    compression="gzip"
)

print(metadata.head())
print(metadata.shape)

raw_dir = Path("GSE164378_RAW")  # ggf. anpassen

matrix_path = raw_dir / "GSM5008737_RNA_3P-matrix.mtx.gz"
barcodes_path = raw_dir / "GSM5008737_RNA_3P-barcodes.tsv.gz"
features_path = raw_dir / "GSM5008737_RNA_3P-features.tsv.gz"

# load the 3P matrix and transpose it to have cells x genes and convert iot to a sparsematrix
X = scipy.io.mmread(matrix_path).T.tocsr()

# Load the cellbarcodes
with gzip.open(barcodes_path, "rt") as f:
    barcodes = [line.strip() for line in f]

# load the genes
features = pd.read_csv(
    features_path,
    sep="\t",
    header=None,
    compression="gzip"
)

gene_names = features[0].values

# Create the AnnData Object
adata_rna = ad.AnnData(
    X=X,
    obs=pd.DataFrame(index=barcodes),
    var=pd.DataFrame(index=gene_names)
)

#Repeat with the ADT Data
matrix_path = raw_dir / "GSM5008738_ADT_3P-matrix.mtx.gz"
barcodes_path = raw_dir / "GSM5008738_ADT_3P-barcodes.tsv.gz"
features_path = raw_dir / "GSM5008738_ADT_3P-features.tsv.gz"

X_adt = scipy.io.mmread(matrix_path).T.tocsr()

with gzip.open(barcodes_path, "rt") as f:
    adt_barcodes = [line.strip() for line in f]

adt_features = pd.read_csv(
    features_path,
    sep="\t",
    header=None,
    compression="gzip"
)

protein_names = adt_features[0].values

adata_adt = ad.AnnData(
    X=X_adt,
    obs=pd.DataFrame(index=adt_barcodes),
    var=pd.DataFrame(index=protein_names)
)

#Merge the meta data into the anndata object
adata_rna.obs = metadata.loc[adata_rna.obs_names].copy()
adata_rna.obs.head()

adata_adt.obs = metadata.loc[adata_adt.obs_names].copy()
adata_adt.obs.head()

                     nCount_ADT  nFeature_ADT  nCount_RNA  nFeature_RNA  \
L1_AAACCCAAGAAACTCA        7535           217       10823          2915   
L1_AAACCCAAGACATACA        6013           209        5864          1617   
L1_AAACCCACAACTGGTT        6620           213        5067          1381   
L1_AAACCCACACGTACTA        3567           202        4786          1890   
L1_AAACCCACAGCATACT        6402           215        6505          1621   

                        orig.ident lane donor  time celltype.l1 celltype.l2  \
L1_AAACCCAAGAAACTCA  SeuratProject   L1    P2     7        Mono   CD14 Mono   
L1_AAACCCAAGACATACA  SeuratProject   L1    P1     7       CD4 T     CD4 TCM   
L1_AAACCCACAACTGGTT  SeuratProject   L1    P4     2       CD8 T   CD8 Naive   
L1_AAACCCACACGTACTA  SeuratProject   L1    P3     7          NK          NK   
L1_AAACCCACAGCATACT  SeuratProject   L1    P4     7       CD8 T   CD8 Naive   

                    celltype.l3 Phase   Batch  
L1_AAACCCAAGAAACTCA   CD14

,nCount_ADT,nFeature_ADT,nCount_RNA,nFeature_RNA,orig.ident,lane,donor,time,celltype.l1,celltype.l2,celltype.l3,Phase,Batch
L1_AAACCCAAGAAACTCA,7535,217,10823,2915,SeuratProject,L1,P2,7,Mono,CD14 Mono,CD14 Mono,G1,Batch1
L1_AAACCCAAGACATACA,6013,209,5864,1617,SeuratProject,L1,P1,7,CD4 T,CD4 TCM,CD4 TCM_1,G1,Batch1
L1_AAACCCACAACTGGTT,6620,213,5067,1381,SeuratProject,L1,P4,2,CD8 T,CD8 Naive,CD8 Naive,S,Batch1
L1_AAACCCACACGTACTA,3567,202,4786,1890,SeuratProject,L1,P3,7,NK,NK,NK_2,G1,Batch1
L1_AAACCCACAGCATACT,6402,215,6505,1621,SeuratProject,L1,P4,7,CD8 T,CD8 Naive,CD8 Naive,G1,Batch1


# 2.) Split data in train and testset

In [27]:
train_idx, test_idx = train_test_split(
    adata_rna.obs_names,
    test_size=0.2,
    random_state=42,
    stratify=adata_rna.obs["celltype.l1"]  # Not sure which one suits us best: Could also change to l2 or l3
)

adata_rna_train = adata_rna[train_idx].copy()
adata_rna_test = adata_rna[test_idx].copy()

adata_adt_train = adata_adt[train_idx].copy()
adata_adt_test = adata_adt[test_idx].copy()

#See if the test and train split works
la = adata_rna_train.obs_names == adata_adt_train.obs_names
print(np.where(la == False)) 
la = adata_rna_test.obs_names == adata_adt_test.obs_names
print(np.where(la == False)) 

(array([], dtype=int64),)
(array([], dtype=int64),)


# 3.) Preprocessing and Feature selection

In [28]:
#For the train and test split normalize Counts and log transform
sp.pp.normalize_total(adata_rna_train)
sp.pp.log1p(adata_rna_train)

sp.pp.normalize_total(adata_rna_test)
sp.pp.log1p(adata_rna_test)

#Feature Selection = HVG selection on train data to avoid data leakage
sp.pp.highly_variable_genes(adata_rna_train)
adata_rna_hvg_train = adata_rna_train[:, adata_rna_train.var.highly_variable]
adata_rna_hvg_test = adata_rna_test[:, adata_rna_train.var.highly_variable]

print("Number of genes: ",adata_rna_train.n_vars)
print("Number of high variable genes train dataset: ", adata_rna_hvg_train.n_vars)
print("Number of high variable genes test testdataset: ", adata_rna_hvg_test.n_vars)

Number of genes:  33538
Number of high variable genes train dataset:  1588
Number of high variable genes test testdataset:  1588


In [29]:
len(adata_rna_hvg_train.X.toarray())

129411

In [30]:
len(adata_rna_hvg_test.X.toarray())

32353

In [31]:
#Use muon to CLR transform ADT counts
import muon as mu
# I wan do tave the raw counts. Maybe we can use them later. They are saved in the "layer" Property of the anndata project
adata_adt_train.layers["raw_counts"] = adata_adt_train.X.copy()
adata_adt_test.layers["raw_counts"] = adata_adt_test.X.copy()

#ADT counts are saved as int64 and we need to c0onvert them to float64 for CLR transofrmation
adata_adt_train.X = adata_adt_train.X.astype(np.float64)
adata_adt_test.X = adata_adt_test.X.astype(np.float64)

mu.prot.pp.clr(adata_adt_train)
mu.prot.pp.clr(adata_adt_test)

c:\Users\Sören\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\muon\_prot\preproc.py:219: UserWarning: adata.X is sparse but not in CSC format. Converting to CSC.
  warn("adata.X is sparse but not in CSC format. Converting to CSC.")


In [32]:
#Tranform whole ADT Dataset for ground truth
print(adata_adt.X[:5,:5].toarray())
adata_adt.X = adata_adt.X.astype(np.float64)
mu.prot.pp.clr(adata_adt)
print(adata_adt.X[:5,:5].toarray())

[[ 66  15  37 142   4]
 [  4  13   9  66  34]
 [  6  19   8  94  10]
 [  5   5   6  79  10]
 [  4  10  14 115  10]]
[[2.13422086 1.09825018 1.19774723 0.77463608 0.27493457]
 [0.37262714 1.00517743 0.44630981 0.43418922 1.30556645]
 [0.51719402 1.26185227 0.40548613 0.57344892 0.58283629]
 [0.44752076 0.51060838 0.31847093 0.50126337 0.58283629]
 [0.37262714 0.84698749 0.6286381  0.66648238 0.58283629]]


# 3.) Setting up the ground truth

In [33]:
adata_adt.obs

,nCount_ADT,nFeature_ADT,nCount_RNA,nFeature_RNA,orig.ident,lane,donor,time,celltype.l1,celltype.l2,celltype.l3,Phase,Batch
L1_AAACCCAAGAAACTCA,7535,217,10823,2915,SeuratProject,L1,P2,7,Mono,CD14 Mono,CD14 Mono,G1,Batch1
L1_AAACCCAAGACATACA,6013,209,5864,1617,SeuratProject,L1,P1,7,CD4 T,CD4 TCM,CD4 TCM_1,G1,Batch1
L1_AAACCCACAACTGGTT,6620,213,5067,1381,SeuratProject,L1,P4,2,CD8 T,CD8 Naive,CD8 Naive,S,Batch1
L1_AAACCCACACGTACTA,3567,202,4786,1890,SeuratProject,L1,P3,7,NK,NK,NK_2,G1,Batch1
L1_AAACCCACAGCATACT,6402,215,6505,1621,SeuratProject,L1,P4,7,CD8 T,CD8 Naive,CD8 Naive,G1,Batch1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
E2L8_TTTGTTGGTCGTGATT,4170,197,9346,2201,SeuratProject,E2L8,P5,7,CD8 T,CD8 Naive,CD8 Naive,S,Batch2
E2L8_TTTGTTGGTGTGCCTG,6927,209,9318,2938,SeuratProject,E2L8,P5,2,Mono,CD14 Mono,CD14 Mono,G1,Batch2
E2L8_TTTGTTGGTTAGTTCG,4222,206,11619,3224,SeuratProject,E2L8,P8,0,B,B intermediate,B intermediate kappa,S,Batch2
E2L8_TTTGTTGGTTGGCTAT,6063,217,15436,3999,SeuratProject,E2L8,P5,2,Mono,CD16 Mono,CD16 Mono,G1,Batch2


In [34]:
adata_adt_train.var_names

Index(['CD39', 'Rat-IgG1-1', 'CD107a', 'CD62P', 'TCR-2', 'CD30', 'CD31',
       'CD34', 'CD35', 'CD36',
       ...
       'CD169', 'CD28', 'CD161', 'CD163', 'CD138-1', 'CD164', 'CD138-2',
       'CD144', 'CD202b', 'CD11c'],
      dtype='object', length=228)

In [35]:
#Convert the ADT Anndata Object into a DataFrame for the one vs Rest test
adt_df = pd.DataFrame(adata_adt.X.toarray(),index=adata_adt.obs_names, columns=adata_adt.var_names)
adt_df

,CD39,Rat-IgG1-1,CD107a,CD62P,TCR-2,CD30,CD31,CD34,CD35,CD36,...,CD169,CD28,CD161,CD163,CD138-1,CD164,CD138-2,CD144,CD202b,CD11c
L1_AAACCCAAGAAACTCA,2.134221,1.098250,1.197747,0.774636,0.274935,0.722762,1.433640,0.848614,2.136687,1.635849,...,0.945264,0.912245,0.692939,1.346169,0.631639,0.846985,0.796635,0.785281,1.287311,1.356701
L1_AAACCCAAGACATACA,0.372627,1.005177,0.446310,0.434189,1.305566,0.722762,0.457878,0.555291,0.448723,0.000000,...,0.814831,1.369084,1.609104,1.011361,0.533302,0.236273,0.340764,0.714857,0.837904,0.819198
L1_AAACCCACAACTGGTT,0.517194,1.261852,0.405486,0.573449,0.582836,1.430395,0.858409,0.466220,0.467573,0.464077,...,0.945264,1.021830,0.000000,0.838115,0.631639,0.236273,1.015023,0.714857,0.717456,0.559429
L1_AAACCCACACGTACTA,0.447521,0.510608,0.318471,0.501263,0.582836,0.000000,0.583268,0.712669,0.442360,0.258721,...,0.273607,0.648760,1.298980,1.158978,0.803331,0.000000,0.911779,0.369758,0.837904,0.559429
L1_AAACCCACAGCATACT,0.372627,0.846987,0.628638,0.666482,0.582836,0.881073,1.172630,0.637073,0.545386,0.906790,...,0.814831,1.166529,0.000000,0.197758,0.533302,0.236273,0.964733,0.557121,0.995091,0.525915
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
E2L8_TTTGTTGGTCGTGATT,0.106956,0.236274,0.523274,0.396118,0.852238,0.302599,0.522537,0.637073,0.213858,0.258721,...,0.000000,0.810752,0.000000,0.362791,0.533302,0.587543,0.239553,0.557121,0.837904,0.339091
E2L8_TTTGTTGGTGTGCCTG,1.876622,0.846987,1.216440,1.134282,0.626057,0.302599,1.247389,0.637073,1.817493,1.635849,...,2.217263,0.570405,0.000000,1.346169,0.803331,1.369078,0.796635,0.557121,0.580488,1.822943
E2L8_TTTGTTGGTTAGTTCG,1.310832,0.510608,0.594735,0.496261,0.274935,0.881073,0.397654,0.555291,2.246535,0.779825,...,0.273607,0.514535,0.287578,0.738760,0.533302,1.964307,0.516829,0.261022,0.232938,0.623266
E2L8_TTTGTTGGTTGGCTAT,1.767140,0.427255,0.916329,1.213226,0.490351,0.722762,1.220492,1.124557,0.669769,0.779825,...,1.687469,0.648760,1.203681,1.011361,0.803331,0.427253,1.015023,0.639096,0.651315,1.672421


In [36]:
adt_df.values

array([[2.13422086, 1.09825018, 1.19774723, ..., 0.78528099, 1.28731131,
        1.35670115],
       [0.37262714, 1.00517743, 0.44630981, ..., 0.7148574 , 0.83790364,
        0.81919775],
       [0.51719402, 1.26185227, 0.40548613, ..., 0.7148574 , 0.71745577,
        0.55942948],
       ...,
       [1.31083189, 0.51060838, 0.59473538, ..., 0.26102165, 0.23293801,
        0.62326559],
       [1.76714013, 0.42725511, 0.91632858, ..., 0.63909604, 0.65131493,
        1.67242065],
       [1.47854821, 0.65898343, 1.07760048, ..., 0.46782096, 0.23293801,
        1.41440319]], shape=(161764, 228))

In [37]:
#Test one vs Rest an use The wilcoxon ranksum test in scipy 
from scipy.stats import ranksums
rows = []
#Lets loop through every cell type
for celltype in adata_adt.obs["celltype.l1"].unique(): #Here we can change the level of granularity.
    
    #Filter for the celltype we are looking at
    filtered_for_celltype = adata_adt.obs["celltype.l1"] == celltype
    
    #For every celltype we will loop through every surface_protein
    for surface_protein in adata_adt_train.var_names:
        
        adt_in_celltype = adt_df.loc[filtered_for_celltype.values, surface_protein]
        adt_in_rest = adt_df.loc[~filtered_for_celltype.values, surface_protein]
        
        #Now we will do the ranksums test
        score_wilc_ranksum_test, p_val = ranksums(adt_in_celltype,adt_in_rest)
        
        #To filter later, we also need the mean difference
        mean_difference = adt_in_celltype.mean() - adt_in_rest.mean()
        
        #Now we append the row to rows
        rows.append({"Celltype":celltype, "Surface_Protein": surface_protein, "Score_Wilcox_Ranksum": score_wilc_ranksum_test,"P_Value":p_val, "Mean_Difference": mean_difference })

surface_protein_markeer_df = pd.DataFrame(rows)
surface_protein_markeer_df

,Celltype,Surface_Protein,Score_Wilcox_Ranksum,P_Value,Mean_Difference
0,Mono,CD39,270.987312,0.000000e+00,0.939088
1,Mono,Rat-IgG1-1,66.712881,0.000000e+00,0.135797
2,Mono,CD107a,244.492463,0.000000e+00,0.563157
3,Mono,CD62P,287.380065,0.000000e+00,0.831591
4,Mono,TCR-2,-94.264277,0.000000e+00,-0.178552
...,...,...,...,...,...
1819,DC,CD164,18.902758,1.082444e-79,0.108118
1820,DC,CD138-2,-7.671698,1.697336e-14,-0.043890
1821,DC,CD144,-2.216141,2.668188e-02,-0.007877
1822,DC,CD202b,-4.502970,6.701021e-06,-0.026231


### We have to adjust for multiple Testing

In [43]:
from statsmodels.stats.multitest import multipletests

surface_protein_markeer_df["p_adj"] = (surface_protein_markeer_df.groupby("Celltype")["P_Value"].transform(lambda p: multipletests(p, method="fdr_bh")[1]))

In [45]:
#Now we filter the Proteins which are are significantly more present on a specific celltype -> A WicoxRankSum Score > 0 and a mean diff > 0
surface_protein_markeer_df = surface_protein_markeer_df[(surface_protein_markeer_df["Score_Wilcox_Ranksum"] > 0) & (surface_protein_markeer_df["Mean_Difference"] > 0) & (surface_protein_markeer_df["p_adj"] < 0.05)].copy() #Should we also Check for p_values < 0.05?
surface_protein_markeer_df

,Celltype,Surface_Protein,Score_Wilcox_Ranksum,P_Value,Mean_Difference,p_adj
0,Mono,CD39,270.987312,0.000000e+00,0.939088,0.000000e+00
1,Mono,Rat-IgG1-1,66.712881,0.000000e+00,0.135797,0.000000e+00
2,Mono,CD107a,244.492463,0.000000e+00,0.563157,0.000000e+00
3,Mono,CD62P,287.380065,0.000000e+00,0.831591,0.000000e+00
5,Mono,CD30,55.247717,0.000000e+00,0.127956,0.000000e+00
...,...,...,...,...,...,...
1813,DC,CD119,40.210768,0.000000e+00,0.263888,0.000000e+00
1814,DC,CD169,24.833501,3.898509e-136,0.243248,7.047305e-136
1817,DC,CD163,21.399842,1.340624e-101,0.137727,2.065879e-101
1819,DC,CD164,18.902758,1.082444e-79,0.108118,1.615074e-79


### We coudl think about filtering just the top 10 surface proteins per celltype

In [46]:
top_surface_proteins = (
    surface_protein_markeer_df
    .sort_values(["Score_Wilcox_Ranksum"], ascending=[False])
    .groupby("Celltype")
    .head(10)
    .reset_index(drop=True)
)
top_surface_proteins

,Celltype,Surface_Protein,Score_Wilcox_Ranksum,P_Value,Mean_Difference,p_adj
0,Mono,CLEC12A,313.102753,0.0,2.048164,0.0
1,Mono,CD86,310.891755,0.0,1.312988,0.0
2,Mono,CD64,310.853268,0.0,1.624797,0.0
3,Mono,CD155,310.319868,0.0,1.353461,0.0
4,Mono,CD11c,309.026098,0.0,1.130338,0.0
...,...,...,...,...,...,...
75,other T,CD45RB,51.502767,0.0,0.267611,0.0
76,other T,CD314,49.655158,0.0,0.268227,0.0
77,other T,CD96,45.833375,0.0,0.220501,0.0
78,other T,TCR-V-7.2,45.420161,0.0,0.809956,0.0


In [47]:
print(top_surface_proteins["Surface_Protein"].unique())
print(len(top_surface_proteins["Surface_Protein"].unique()))

['CLEC12A' 'CD86' 'CD64' 'CD155' 'CD11c' 'CD11b-2' 'CD93' 'CD172a' 'CD31'
 'CD18' 'CD4-1' 'CD4-2' 'CD109' 'CD8a' 'CD8' 'CD3-2' 'CD28' 'CD3-1'
 'CD127' 'CD56-2' 'CD27' 'CD278' 'TCR-2' 'CD314' 'CD19' 'CD16' 'CD335'
 'CD268' 'CD22' 'CD21' 'CD20' 'CD72' 'CD122' 'CD196' 'CD2' 'CD56-1'
 'CD275-2' 'IgD' 'CD45RA' 'CD272' 'CD45RB' 'CD337' 'CD244' 'CD43' 'CD96'
 'CD110' 'CD71' 'HLA-DR' 'CD49b' 'CD161' 'CD42b' 'CD41' 'CLEC2' 'CD69'
 'CD102' 'CD271' 'CD112' 'CD9' 'CD61' 'Integrin-7' 'CD123' 'CD1c' 'CD49d'
 'CD54' 'CD141' 'CD195' 'TCR-V-7.2' 'CD81']
68


### That Query gets us the genes to the surface Proteins from the UniProt Database

In [99]:
import requests
from io import StringIO

def search_uniprot_protein_to_gene(protein_names, organism_id=9606, reviewed_only=True):
    all_results = []

    for protein in protein_names:
        query = f'(protein_name:"{protein}" OR gene:{protein}) AND organism_id:{organism_id}'
        
        if reviewed_only:
            query += " AND reviewed:true"

        url = "https://rest.uniprot.org/uniprotkb/search"

        params = {
            "query": query,
            "fields": "accession,id,protein_name,gene_names,organism_name,reviewed",
            "format": "tsv",
            "size": 10
        }

        r = requests.get(url, params=params)
        r.raise_for_status()

        if r.text.strip():
            df = pd.read_csv(StringIO(r.text), sep="\t")
            df.insert(0, "Surface_Protein", protein)
            all_results.append(df)
        else:
            all_results.append(pd.DataFrame({
                "Surface_Protein": [protein],
                "Entry": [None],
                "Gene Names": [None]
            }))

    return pd.concat(all_results, ignore_index=True)



ground_truth_df = search_uniprot_protein_to_gene(top_surface_proteins["Surface_Protein"].unique())

ground_truth_df

,Surface_Protein,Entry,Entry Name,Protein names,Gene Names,Organism,Reviewed
0,CLEC12A,Q5QGZ9,CL12A_HUMAN,C-type lectin domain family 12 member A (C-typ...,CLEC12A CLL1 DCAL2 KLRL1 MICL,Homo sapiens (Human),reviewed
1,CD86,P42081,CD86_HUMAN,T-lymphocyte activation antigen CD86 (Activati...,CD86 CD28LG2,Homo sapiens (Human),reviewed
2,CD64,P12314,FCGR1_HUMAN,High affinity immunoglobulin gamma Fc receptor...,FCGR1A FCG1 FCGR1 IGFR1,Homo sapiens (Human),reviewed
3,CD155,P15151,PVR_HUMAN,Poliovirus receptor (Nectin-like protein 5) (N...,PVR PVS,Homo sapiens (Human),reviewed
4,CD11c,P20702,ITAX_HUMAN,Integrin alpha-X (CD11 antigen-like family mem...,ITGAX CD11C,Homo sapiens (Human),reviewed
...,...,...,...,...,...,...,...
74,CD54,P05362,ICAM1_HUMAN,Intercellular adhesion molecule 1 (ICAM-1) (Ma...,ICAM1,Homo sapiens (Human),reviewed
75,CD141,P07204,TRBM_HUMAN,Thrombomodulin (TM) (Fetomodulin) (CD antigen ...,THBD THRM,Homo sapiens (Human),reviewed
76,CD195,P51681,CCR5_HUMAN,C-C chemokine receptor type 5 (C-C CKR-5) (CC-...,CCR5 CMKBR5,Homo sapiens (Human),reviewed
77,CD81,P60033,CD81_HUMAN,CD81 antigen (26 kDa cell surface protein TAPA...,CD81 TAPA1 TSPAN28,Homo sapiens (Human),reviewed


In [103]:
gt_merged = top_surface_proteins[["Celltype", "Surface_Protein"]].drop_duplicates().merge(
    ground_truth_df[["Surface_Protein", "Gene Names"]],
    on="Surface_Protein",
    how="left"
)

gt_merged.head()

,Celltype,Surface_Protein,Gene Names
0,Mono,CLEC12A,CLEC12A CLL1 DCAL2 KLRL1 MICL
1,Mono,CD86,CD86 CD28LG2
2,Mono,CD64,FCGR1A FCG1 FCGR1 IGFR1
3,Mono,CD155,PVR PVS
4,Mono,CD11c,ITGAX CD11C


# 4.) Spearman-Correlation

In [ ]:
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler

#lets just implement it for 1 specific celltype
def ranking_genes_with_spearman(data, celltype, celltype_granularity = "celltype.l1"):
    
    #conver the sparse matrix
    data_array = data.X.toarray()
    print("Data Array: ", data_array)
    #Get the Labels
    labels = data.obs[celltype_granularity].values
    genes = np.array(data.var_names)

    #Create our y variable -> one = desired celltype = 1 and the other celltypes = 0
    y = (labels == celltype).astype(float)
    print("Y-variable: ", y)
        
    #Some Debugging Prints
    print("X shape:", X.shape)
    print("labels shape:", labels.shape)
    print("genes shape:", genes.shape)
    print("y shape:", y.shape)
    print("number target cells:", y.sum())
    print("number rest cells:", len(y) - y.sum())
    
    """
    We could now use scipys spearman correlation function spearm(), but that might take a while because we would need to loop though every of the 1588 genes and apply spearm.
    We can also standardize the ranks and do a matrix multiplikation instead.
    """
    # Spearman needs ranks -> rank the expression levels. Use scipys rankdata with axis = 0 for that (every column = gene)
    ranked_x = np.apply_along_axis(rankdata, 0, adata_rna_hvg_train.X.toarray(), method="average")
    print("X_ranked shape:", ranked_x.shape)
    
    #We use scikitlearns standardscaler to get mu=0 and sd = 1
    ranked_x = StandardScaler().fit_transform(ranked_x)
    
    #We also do it for y
    y_ranked = rankdata(y, method="average")
    y_ranked = (y_ranked - y_ranked.mean()) / y_ranked.std()
    
    print("y_ranked mean:", y_ranked.mean())
    print("y_ranked std:", y_ranked.std())
    
    #Now Just calculate the scores with the matmult formula for spearman
    n = ranked_x.shape[0]
    scores = ranked_x.T @ y_ranked / n
    
    #Create a DataFrame that the function can return
    ranking = pd.DataFrame({"gene": genes,"score": scores,"celltype": celltype})
    
    #Sort the DataFrame and add direct rank
    ranking = ranking.sort_values("score", ascending=False).reset_index(drop=True)
    ranking["rank"] = np.arange(1, len(ranking) + 1)

    return ranking
    

### Testrun with B Celltype

In [ ]:
spearman_b = ranking_genes_with_spearman(adata_rna_hvg_train,celltype="B")

spearman_b.head(20)

Data Array:  [[0.         0.91866785 0.         ... 0.         0.5613143  0.        ]
 [0.         0.60320175 0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.90923774 0.         ... 0.         0.90923774 0.        ]
 [0.         0.9028097  0.         ... 0.         0.9028097  0.        ]
 [0.         0.         0.         ... 0.6042463  0.6042463  0.        ]]
Y-variable:  [0. 0. 0. ... 0. 0. 0.]
X shape: (161764, 33538)
labels shape: (129411,)
genes shape: (1588,)
y shape: (129411,)
number target cells: 11040.0
number rest cells: 118371.0
X_ranked shape: (129411, 1588)
y_ranked mean: 1.603252264814809e-17
y_ranked std: 1.0


,gene,score,celltype,rank
0,CD79A,0.867835,B,1
1,MS4A1,0.845083,B,2
2,BANK1,0.823069,B,3
3,LINC00926,0.796582,B,4
4,TNFRSF13C,0.792133,B,5
5,VPREB3,0.782107,B,6
6,IGHD,0.755146,B,7
7,CD22,0.752666,B,8
8,PAX5,0.729115,B,9
9,RALGPS2,0.687771,B,10


In [104]:
gt_merged[gt_merged["Celltype"]=="B"]

,Celltype,Surface_Protein,Gene Names
29,B,CD19,CD19
33,B,CD268,TNFRSF13C BAFFR BR3
34,B,CD22,CD22 SIGLEC2
35,B,CD21,CR2 C3DR
36,B,CD20,MS4A1 CD20
37,B,CD20,MS4A7 4SPAN2 CD20L4 CFFM4
38,B,CD20,MS4A6A 4SPAN3 CD20L3 MS4A6 CDA01 MSTP090
39,B,CD20,MS4A4A 4SPAN1 CD20L1 MS4A4 HDCME31P
40,B,CD20,MS4A3 CD20L HTM4
41,B,CD20,MS4A10 CD20L7 MS4A9
